# 보관용 공개 사본

당시 팀 프로젝트의 구현 자료입니다. 출력과 개인 공유 경로를 정리했습니다. 데이터·학습 가중치는 포함하지 않으며, 현재 환경에서 전체 실행을 검증하지 않았습니다. 상세한 버전 차이와 실행 조건은 `docs/evidence.md`, `docs/reproduction.md`를 참고하세요.


In [ ]:
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from google.colab import files

FILE_PATH = 'sycophancy_dataset_combined_v4_1.xlsx'
SHEET_NAME = 'data'

if not os.path.exists(FILE_PATH):
    print(f"⚠️ '{FILE_PATH}' 파일이 존재하지 않습니다.")
    print("파일을 업로드해주세요!")
else:
    print(f"📂 데이터 로드 시작: {FILE_PATH} ...")
    df = pd.read_excel(FILE_PATH, sheet_name=SHEET_NAME)


    df = df.dropna(subset=['text', 'sycophancy', 'bias_level', 'severity'])
    df['text'] = df['text'].str.strip()
    df = df.drop_duplicates(subset=['text'], keep='first')

    severity_mapping = {0: 0, 1: 0, 2: 1, 3: 2}
    expected_severity = df['bias_level'].map(severity_mapping)
    mismatch_count = (df['severity'] != expected_severity).sum()

    print("\n--- Severity Validation ---")
    print(f"Mismatch count (Labeler vs Formula): {mismatch_count}")
    if mismatch_count > 0:
        print("⚠️ Warning: Mismatches found. Overwriting severity to enforce strict dataset rules.")
    df['severity'] = expected_severity


    df['sycophancy'] = df['sycophancy'].astype(int)
    df['bias_level'] = df['bias_level'].astype(int)
    df['severity'] = df['severity'].astype(int)


    df['user_prompt'] = df['text'].apply(lambda x: x.split('[AI 답변]', 1)[0].replace('[유저 질문]', '').strip() if '[AI 답변]' in x else x)
    df['ai_response'] = df['text'].apply(lambda x: x.split('[AI 답변]', 1)[1].strip() if '[AI 답변]' in x else "")

    df['stratify_key'] = df['sycophancy'].astype(str) + "_" + df['bias_level'].astype(str)

    train_df, temp_df = train_test_split(df, test_size=0.2, random_state=42, stratify=df['stratify_key'])
    val_df, test_df = train_test_split(temp_df, test_size=0.5, random_state=42, stratify=temp_df['stratify_key'])

    assert train_df["text"].isin(test_df["text"]).sum() == 0, "Error: Data Leakage between Train and Test!"
    assert train_df["text"].isin(val_df["text"]).sum() == 0, "Error: Data Leakage between Train and Val!"
    assert val_df["text"].isin(test_df["text"]).sum() == 0, "Error: Data Leakage between Val and Test!"
    print("\n--- Data Leakage Check --- \n✅ Passed: No overlapping texts across Train/Val/Test splits.")

    print(f"\n--- Split Results ---")
    print(f"Train size: {len(train_df)} | Val size: {len(val_df)} | Test size: {len(test_df)}")

    print("\n--- Train Data Distribution (Sycophancy & Bias Level) ---")
    print(train_df.groupby(['sycophancy', 'bias_level']).size().reset_index(name='Count'))

    train_df = train_df.drop(columns=['stratify_key'])
    val_df = val_df.drop(columns=['stratify_key'])
    test_df = test_df.drop(columns=['stratify_key'])

    train_df.to_csv('sycophancy_train.csv', index=False, encoding='utf-8-sig')
    val_df.to_csv('sycophancy_val.csv', index=False, encoding='utf-8-sig')
    test_df.to_csv('sycophancy_test.csv', index=False, encoding='utf-8-sig')

    print("\n✅ 파일 생성 완료: 'sycophancy_train.csv', 'sycophancy_val.csv', 'sycophancy_test.csv'")

    print("\n📥 생성된 CSV 파일 다운로드를 시작합니다...")
    files.download('sycophancy_train.csv')
    files.download('sycophancy_val.csv')
    files.download('sycophancy_test.csv')